<a href="https://colab.research.google.com/github/oliveirasamuel5959/VRDFormer_VRD/blob/main/notebooks/colab_kaggle_train_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# VRDFormer Training on Colab with the `vidorsmall` Dataset

Trains the full VRDFormer pipeline (Stage 1 + Stage 2) on **vidorsmall** — an 8x smaller,
coverage-stratified subset of VidOR (875 train / 104 val videos, ~4.5 GB) that guarantees all
50 verbs and 80 objects are represented. Data is hosted on Kaggle, so no Google Drive space is needed.

**Prerequisites:**
- Kaggle API key (`kaggle.json`) from kaggle.com/settings -> API -> Create New Token
- The `samuelpatricio/vrdformer-vidor-small` Kaggle dataset (build it locally with
  `python data/make_kaggle_bundle.py --zip` then `kaggle datasets create -p data/vidor_small_bundle --dir-mode zip`)


## 1. Clone Repo & Install Dependencies


In [ ]:
# Clone your fork (replace YOUR_USERNAME if you forked elsewhere)
!git clone https://github.com/oliveirasamuel5959/VRDFormer_VRD.git
%cd VRDFormer_VRD

# Install dependencies (PyTorch + torchvision are pre-installed on Colab)
!pip install decord timm scipy lap -q
!pip install -U 'git+https://github.com/timmeinhardt/cocoapi.git#subdirectory=PythonAPI' -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Number of available GPUs: {torch.cuda.device_count()}')


## 2. Download the vidorsmall Kaggle Dataset


In [ ]:
# Upload your kaggle.json API key
from google.colab import files
files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

# The archive is built so its contents extract straight into data/:
#   data/vidorsmall/, data/metadata/, data/weights/
!kaggle datasets download samuelpatricio/vrdformer-vidor-small
!unzip -q vrdformer-vidor-small.zip -d data
!rm vrdformer-vidor-small.zip
print('Dataset ready.')


## 3. Verify GPU & Data


In [ ]:
!nvidia-smi

import os
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

checks = [
    ('data/vidorsmall/videos', 'dir'),
    ('data/vidorsmall/annotations/train', 'dir'),
    ('data/vidorsmall/annotations/val', 'dir'),
    ('data/vidorsmall/action.txt', 'file'),
    ('data/metadata/vidorsmall_annotations.pkl', 'file'),
    ('data/metadata/vidorsmall_train_frames_stage1.json', 'file'),
    ('data/metadata/vidorsmall_train_frames_stage2.json', 'file'),
    ('data/metadata/vidorsmall_val_frames.json', 'file'),
    ('data/weights/detr-r101-2c7b67e5.pth', 'file'),
    ('configs/vidorsmall_stage1.json', 'file'),
    ('configs/vidorsmall_stage2.json', 'file'),
]

all_ok = True
for path, kind in checks:
    exists = os.path.isdir(path) if kind == 'dir' else os.path.isfile(path)
    size = f' ({os.path.getsize(path) / 1e6:.1f} MB)' if exists and kind == 'file' else ''
    print(f'  {"OK" if exists else "MISSING"} {path}{size}')
    all_ok = all_ok and exists

if all_ok:
    n_train = len(os.listdir('data/vidorsmall/annotations/train'))
    n_val = len(os.listdir('data/vidorsmall/annotations/val'))
    print(f'\nAll checks passed - {n_train} train / {n_val} val videos. Ready to train!')
else:
    print('\nSome files are missing - check the Kaggle download.')


## 4. Stage 1 Training (pair detection + tracking)


In [ ]:
%%time
# Single GPU. For multi-GPU, replace `python main.py` with:
#   python -m torch.distributed.launch --nproc_per_node=N main.py
!python main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidorsmall_stage1.json


### Verify Stage 1 Output


In [ ]:
import os, glob
ckpts = sorted(glob.glob('data/ckpts/vidorsmall_stage1/checkpoint*.pth'))
print(f'Stage 1 checkpoints: {len(ckpts)}')
for ckpt in ckpts:
    print(f'  {ckpt:<60} {os.path.getsize(ckpt) / 1e6:.1f} MB')


## 5. Stage 2 Training (relation classification)


In [ ]:
%%time
# Loads data/ckpts/vidorsmall_stage1/checkpoint0004.pth (set in the config's "pretrain").
# train_clip_sample_ratio=0.25 in the config trains on every 4th clip to fit a Colab session.
!python main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidorsmall_stage2.json


## 6. Evaluation


In [ ]:
!python main.py \
    --eval \
    --dataset_config configs/vidorsmall_stage2.json \
    --resume data/ckpts/vidorsmall_stage2/checkpoint.pth

# NOTE: metrics on the small val split are noisy and the zero-shot numbers are not
# comparable to full-data results (val-minus-train is computed against the small train
# split). Treat these as pipeline sanity checks, not publishing numbers.


## 7. Download Checkpoints


In [ ]:
!zip -r checkpoints.zip data/ckpts/
from google.colab import files
files.download('checkpoints.zip')

print('\nKey files:')
print('  Stage 1: data/ckpts/vidorsmall_stage1/checkpoint0004.pth')
print('  Stage 2: data/ckpts/vidorsmall_stage2/checkpoint.pth')


## Colab Keep-Alive

Colab disconnects after ~90 min idle. Paste this in the browser console (F12) to reconnect:

```javascript
function ClickConnect(){ document.querySelector("colab-connect-button").click() }
setInterval(ClickConnect, 60000)
```
